# Deep-Guard — Generator Attribution Training

**Phase 2, Idea 2.** A SEPARATE model from Stage 1's real/fake detector — Stage 1 (`deepguard_bouncer.pth`) never changes because of anything in this notebook. This one answers a second, narrower question, asked only once Stage 1 has already flagged something as AI-generated: **which of 8 specific tools most likely made it?**

Same discipline as Stage 1's training notebook throughout — disk-safe partial extraction (a full crash was hit once, earlier in this project, downloading everything at once), split-before-oversample (a real train/test leak was caught and fixed the same way), and honest reporting of whatever the confusion matrix actually shows, not just the headline accuracy.

**Before running:** you need a Kaggle account and an API token (`kaggle.json`) — same one used for Stage 1's datasets, if you still have it saved. Get one at kaggle.com → your profile picture → Settings → API → "Create New Token" if you don't.

## Step 1 — Check the GPU and available disk

In [ ]:
!nvidia-smi
!df -h /content

## Step 2 — Install the Kaggle tool

In [ ]:
!pip install -q kaggle==1.6.17

## Step 3 — Credentials and Drive backup

Upload your `kaggle.json` when prompted. Drive backup is strongly recommended for a multi-hour run — if Colab disconnects, the checkpoint survives.

In [ ]:
from google.colab import files
import os

uploaded = files.upload()
os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json", "wb") as f:
    f.write(uploaded["kaggle.json"])
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Kaggle credentials installed.")

In [ ]:
# Strongly recommended for a multi-hour run. If you'd rather not use
# Drive, set USE_DRIVE_BACKUP = False below and skip this cell's mount.
USE_DRIVE_BACKUP = True

if USE_DRIVE_BACKUP:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BACKUP_DIR = "/content/drive/MyDrive/deepguard_attribution_backup"
    os.makedirs(DRIVE_BACKUP_DIR, exist_ok=True)
    print(f"Drive mounted. Backups will go to {DRIVE_BACKUP_DIR}")

## Step 4 — Configuration

**`GENERATOR_CLASSES` must match `model.py`'s list EXACTLY — same order, same names.** This is the single source of truth this whole notebook and the deployed app both depend on; if the two ever drift apart, the checkpoint will load but every prediction will be silently mislabeled.

8 classes, chosen for genuine architectural diversity on both sides of the GAN/diffusion split (see the Phase 2 build plan for the reasoning), not just picked at random from ArtiFact's 25.

In [ ]:
# MUST match app/model.py's GENERATOR_CLASSES exactly.
GENERATOR_CLASSES = [
    # GAN family
    "stylegan2", "pro_gan", "big_gan", "cycle_gan",
    # Diffusion family
    "ddpm", "latent_diffusion", "stable_diffusion", "glide",
]

# ArtiFact images are small (200x200 source resolution), so this is a much
# lighter per-image footprint than Stage 1's full-resolution photos --
# even PER_CLASS_TARGET=3000 (24,000 images total) should stay well
# within a safe disk budget. The interleaved-extraction safety net below
# still applies regardless, the same way it did for Stage 1.
PER_CLASS_TARGET = 3000
MIN_FREE_GB_SAFETY = 15.0

ARTIFACT_SLUG = "awsaf49/artifact-dataset"
DATA_DIR_NAME = "attribution_data"

CHECKPOINT_PATH = "/content/deepguard_attribution.pth"
BATCH_SIZE = 32
INPUT_SIZE = 224

## Step 5 — Download ArtiFact (partial, disk-safe)

ArtiFact's full download is 31.58 GB across 2.5 million images in 33 folders (8 real-image sources + 25 generator classes) — the whole thing does not need to land on disk, only a sample from the 8 folders we actually want.

Same reasoning as Stage 1's `fetch_partial_by_class`, generalized from 2 classes (real/fake) to N: the zip has to download in full first (a zip's central directory needs the whole file before individual members can be read), then only `PER_CLASS_TARGET` images per class get extracted — chosen at random within each class so the sample isn't biased toward however the zip happens to be ordered — with extraction interleaved round-robin across all 8 classes, so if disk space runs low partway through, what's already on disk is still a BALANCED partial sample across every class rather than 8 fully-sampled classes and 0 empty ones.

In [ ]:
import random, shutil, subprocess, zipfile
from pathlib import Path

DATA_DIR = Path("/content") / DATA_DIR_NAME
DATA_DIR.mkdir(exist_ok=True)


def free_gb():
    return shutil.disk_usage("/content").free / 1e9


def fetch_partial_multiclass(slug, target_dir, class_names, per_class_target, min_free_gb):
    """Generalization of Stage 1's fetch_partial_by_class from 2 classes
    to N. See the markdown cell above for why this shape exists at all --
    it is not an aesthetic choice, it is what avoided a repeat of the
    disk crash Stage 1 actually hit."""
    already_extracted = target_dir.exists() and any(
        p for p in target_dir.rglob("*") if p.is_file() and p.suffix.lower() != ".zip"
    )
    if already_extracted:
        print(f"[skip] {slug} already sampled at {target_dir}")
        return

    target_dir.mkdir(parents=True, exist_ok=True)

    existing_zips = list(target_dir.glob("*.zip"))
    if existing_zips:
        zip_path = existing_zips[0]
        print(f"[skip download] zip already present: {zip_path.name}")
    else:
        print(f"[download] {slug}  (free disk: {free_gb():.1f} GB) -- this is a 31.58 GB zip, expect it to take a while")
        subprocess.run(["kaggle", "datasets", "download", "-d", slug, "-p", str(target_dir)], check=True)
        zip_path = next(target_dir.glob("*.zip"))
        print(f"[downloaded] free disk: {free_gb():.1f} GB")

    print(f"[inspect] reading {slug}'s file list (metadata only, nothing extracted yet)")
    with zipfile.ZipFile(zip_path) as zf:
        names = [n for n in zf.namelist() if not n.endswith("/")]

        candidates_by_class = {}
        for cls in class_names:
            matches = [n for n in names if any(part.lower() == cls for part in Path(n).parts)]
            random.shuffle(matches)
            candidates_by_class[cls] = iter(matches)
            print(f"  {cls}: {len(matches):,} candidates in zip")

        extracted_counts = {cls: 0 for cls in class_names}
        stopped_early = False

        while any(extracted_counts[c] < per_class_target for c in class_names):
            if free_gb() < min_free_gb:
                stopped_early = True
                break
            made_progress = False
            for cls in class_names:
                if extracted_counts[cls] >= per_class_target:
                    continue
                name = next(candidates_by_class[cls], None)
                if name is None:
                    continue
                zf.extract(name, target_dir)
                extracted_counts[cls] += 1
                made_progress = True
            if not made_progress:
                break  # every class's candidate pool is exhausted

    zip_path.unlink()
    status = "(stopped early -- low disk headroom)" if stopped_early else "(reached target for every class)"
    print(f"\n[done] {status}")
    for cls in class_names:
        print(f"  {cls}: {extracted_counts[cls]:,} images extracted")
    print(f"  zip deleted, free disk: {free_gb():.1f} GB")


fetch_partial_multiclass(
    ARTIFACT_SLUG, DATA_DIR, GENERATOR_CLASSES,
    per_class_target=PER_CLASS_TARGET, min_free_gb=MIN_FREE_GB_SAFETY,
)

## Step 6 — Verify what actually got extracted

Same instinct as Stage 1's Step 6: don't assume the download worked as hoped, look at what's actually on disk before building anything on top of it. If disk safety stopped extraction early, some classes may have fewer images than `PER_CLASS_TARGET` — that's fine, the split step below balances against whatever the smallest class actually has, not against the target number.

In [ ]:
IMAGE_EXT = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

class_files = {}
for cls in GENERATOR_CLASSES:
    cls_dir = DATA_DIR / cls
    files_found = [p for p in cls_dir.rglob("*") if p.suffix.lower() in IMAGE_EXT] if cls_dir.exists() else []
    class_files[cls] = files_found
    print(f"  {cls:<18} {len(files_found):,} images")

min_class_size = min(len(v) for v in class_files.values())
print(f"\nSmallest class: {min_class_size:,} images -- this caps how many images per class the balanced split below can use.")
assert min_class_size >= 100, "A class has under 100 images -- something went wrong with extraction, check Step 5's output above before continuing." 

## Step 7 — The label-order safeguard

Identical purpose to Stage 1's Step 7: `GENERATOR_CLASSES`' index position IS the label everywhere in this notebook, in `model.py`, and in the deployed app. This assertion exists so a future edit that reorders the list gets caught here, immediately, in a cell that fails loudly — not three hours into training, or worse, silently, as a checkpoint that loads fine but has every prediction shifted by one class.

In [ ]:
EXPECTED_GENERATOR_CLASSES = [
    "stylegan2", "pro_gan", "big_gan", "cycle_gan",
    "ddpm", "latent_diffusion", "stable_diffusion", "glide",
]
assert GENERATOR_CLASSES == EXPECTED_GENERATOR_CLASSES, (
    "GENERATOR_CLASSES has drifted from the copy in this cell (which must itself match "
    "app/model.py). Fix whichever one is wrong before continuing -- do not train against "
    "a class order that doesn't match the deployed app."
)
print("Label order confirmed consistent.")

## Step 8 — Build a balanced train / validation / test split

**Split on unique files FIRST, balance/cap per class AFTER.** This is the exact discipline that fixed a real train/test leak earlier in this project (oversampling before splitting let duplicate files land on both sides of the split). There is no oversampling in this notebook at all — every class is capped to the same size (`min_class_size` from Step 6), so there's nothing to leak in the first place — but the split-before-touching-anything-else order is kept identical on principle, since it costs nothing and removes a whole category of mistake by construction.

In [ ]:
random.seed(42)

TRAIN_FRACTION, VALID_FRACTION = 0.75, 0.10  # remainder (0.15) is test

train_files, valid_files, test_files = [], [], []
train_labels, valid_labels, test_labels = [], [], []

for class_idx, cls in enumerate(GENERATOR_CLASSES):
    files_for_class = class_files[cls][:min_class_size]  # cap every class to the same size -- balanced by construction
    random.shuffle(files_for_class)

    n = len(files_for_class)
    n_train = int(n * TRAIN_FRACTION)
    n_valid = int(n * VALID_FRACTION)

    train_files += files_for_class[:n_train]
    train_labels += [class_idx] * n_train
    valid_files += files_for_class[n_train:n_train + n_valid]
    valid_labels += [class_idx] * n_valid
    test_files += files_for_class[n_train + n_valid:]
    test_labels += [class_idx] * (n - n_train - n_valid)

print(f"Train: {len(train_files):,}  Valid: {len(valid_files):,}  Test: {len(test_files):,}")

# The actual leak-proof check, not just a hope: every file path is unique
# to exactly one split.
train_set, valid_set, test_set = set(train_files), set(valid_files), set(test_files)
assert not (train_set & valid_set), "Leak: a file appears in both train and valid."
assert not (train_set & test_set), "Leak: a file appears in both train and test."
assert not (valid_set & test_set), "Leak: a file appears in both valid and test."
print("No overlap between splits -- confirmed, not assumed.")

## Step 9 — Dataset and DataLoader

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

NORMALIZE_MEAN = [0.485, 0.456, 0.406]
NORMALIZE_STD = [0.229, 0.224, 0.225]

train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop(INPUT_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
])
eval_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop(INPUT_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORMALIZE_MEAN, std=NORMALIZE_STD),
])


class GeneratorDataset(Dataset):
    def __init__(self, file_paths, labels, transform):
        self.file_paths = file_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        image = Image.open(self.file_paths[idx]).convert("RGB")
        return self.transform(image), self.labels[idx]


train_loader = DataLoader(GeneratorDataset(train_files, train_labels, train_transform),
                           batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valid_loader = DataLoader(GeneratorDataset(valid_files, valid_labels, eval_transform),
                           batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
test_loader = DataLoader(GeneratorDataset(test_files, test_labels, eval_transform),
                          batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

## Step 10 — Build the model

**Duplicated from `app/model.py`'s `build_attribution_model()`, line for line — same discipline as Stage 1.** The two environments don't share a Python import path, so this has to be declared twice. If you ever change this cell, change `model.py` identically and retrain, or the saved weights will fail to load, or load into the wrong layer shapes silently.

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights


def build_attribution_model(pretrained=False):
    weights = EfficientNet_B0_Weights.DEFAULT if pretrained else None
    model = efficientnet_b0(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, len(GENERATOR_CLASSES))
    return model


model = build_attribution_model(pretrained=True).to(device)
print(f"Model built: EfficientNet-B0, {len(GENERATOR_CLASSES)}-class head")

## Step 11 — Training loop

In [ ]:
import time

criterion = nn.CrossEntropyLoss()  # multi-class -- expects raw logits + integer class labels, applies log-softmax internally


def freeze_backbone(m):
    for param in m.features.parameters():
        param.requires_grad = False


def unfreeze_everything(m):
    for param in m.parameters():
        param.requires_grad = True


def run_epoch(loader, model, optimizer=None):
    is_training = optimizer is not None
    model.train(is_training)

    total_loss, total_correct, total_seen = 0.0, 0, 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        if is_training:
            optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        if is_training:
            loss.backward()
            optimizer.step()

        total_loss += loss.item() * images.size(0)
        total_correct += (logits.argmax(dim=1) == labels).sum().item()
        total_seen += images.size(0)

    return total_loss / total_seen, total_correct / total_seen

## Step 12 — Phase A: warm up the classifier head

Same two-phase approach as Stage 1: freeze the pretrained backbone first, train only the new classification head, so the randomly-initialized head doesn't send large, disruptive gradients back through the pretrained ImageNet features before it has learned anything sensible.

In [ ]:
freeze_backbone(model)
optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-3)

PHASE_A_EPOCHS = 3
for epoch in range(PHASE_A_EPOCHS):
    start = time.time()
    train_loss, train_acc = run_epoch(train_loader, model, optimizer)
    valid_loss, valid_acc = run_epoch(valid_loader, model)
    print(f"[Phase A] epoch {epoch+1}/{PHASE_A_EPOCHS}  "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f}  "
          f"valid_loss={valid_loss:.4f} valid_acc={valid_acc:.4f}  "
          f"({time.time()-start:.0f}s)")

## Step 13 — Phase B: fine-tune the whole network

Unfreeze everything, drop the learning rate, and use a cosine schedule -- the same recipe Stage 1 used, chosen there for the same reason it applies here: fine-tuning the whole backbone needs smaller, decaying steps than head-only warmup, or it can undo the pretrained features' useful structure instead of adapting them.

In [ ]:
unfreeze_everything(model)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

PHASE_B_EPOCHS = 8
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PHASE_B_EPOCHS)

best_valid_acc = 0.0
for epoch in range(PHASE_B_EPOCHS):
    start = time.time()
    train_loss, train_acc = run_epoch(train_loader, model, optimizer)
    valid_loss, valid_acc = run_epoch(valid_loader, model)
    scheduler.step()

    improved = valid_acc > best_valid_acc
    if improved:
        best_valid_acc = valid_acc
        torch.save(model.state_dict(), CHECKPOINT_PATH)

    print(f"[Phase B] epoch {epoch+1}/{PHASE_B_EPOCHS}  "
          f"train_loss={train_loss:.4f} train_acc={train_acc:.4f}  "
          f"valid_loss={valid_loss:.4f} valid_acc={valid_acc:.4f}  "
          f"{'(saved -- best so far)' if improved else ''}  ({time.time()-start:.0f}s)")

    if USE_DRIVE_BACKUP:
        try:
            shutil.copy(CHECKPOINT_PATH, f"{DRIVE_BACKUP_DIR}/deepguard_attribution_epoch{epoch+1}.pth")
        except Exception as exc:
            print(f"  [warning] Drive backup failed this epoch (continuing anyway): {exc}")

print(f"\nBest validation accuracy: {best_valid_acc:.4f}")

## Step 14 — Final test on data never seen during training

Loads the BEST checkpoint (by validation accuracy, not just whatever the last epoch happened to produce) and scores it once against the held-out test split.

In [ ]:
model.load_state_dict(torch.load(CHECKPOINT_PATH, map_location=device))
test_loss, test_acc = run_epoch(test_loader, model)
print(f"Test accuracy: {test_acc:.4f}  (test loss: {test_loss:.4f})")

## Step 15 — Detailed breakdown: confusion matrix + family-clustering analysis

An overall accuracy number can hide exactly the kind of thing this whole project has already learned to check for by hand — a per-CLASS breakdown, not just one aggregate. This step adds one honest analysis beyond Stage 1's equivalent: **when the model is wrong, is it confusing GAN-with-GAN and diffusion-with-diffusion (a sign it learned real, generalizable family-level structure), or is it confusing across families more or less at random (a sign it mostly memorized per-class quirks)?** Report whichever one actually happens — this is exactly the kind of finding the project has a track record of documenting honestly rather than papering over.

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

model.eval()
all_preds, all_labels = [], []
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        all_preds += logits.argmax(dim=1).cpu().tolist()
        all_labels += labels.tolist()

print(classification_report(all_labels, all_preds, target_names=GENERATOR_CLASSES, digits=3))

cm = confusion_matrix(all_labels, all_preds)
print("Confusion matrix (rows = true class, columns = predicted class):")
print("        " + " ".join(f"{c[:6]:>7}" for c in GENERATOR_CLASSES))
for i, row in enumerate(cm):
    print(f"{GENERATOR_CLASSES[i][:7]:<8}" + " ".join(f"{v:>7}" for v in row))

In [ ]:
# Family-clustering analysis -- see the markdown cell above for why this
# matters more than the raw confusion matrix numbers alone.
GAN_CLASSES = {"stylegan2", "pro_gan", "big_gan", "cycle_gan"}
DIFFUSION_CLASSES = {"ddpm", "latent_diffusion", "stable_diffusion", "glide"}

def family_of(class_name):
    return "GAN" if class_name in GAN_CLASSES else "Diffusion"

within_family_errors, cross_family_errors, total_errors = 0, 0, 0
for true_idx, pred_idx in zip(all_labels, all_preds):
    if true_idx == pred_idx:
        continue
    total_errors += 1
    true_family = family_of(GENERATOR_CLASSES[true_idx])
    pred_family = family_of(GENERATOR_CLASSES[pred_idx])
    if true_family == pred_family:
        within_family_errors += 1
    else:
        cross_family_errors += 1

print(f"Total misclassifications: {total_errors}")
if total_errors > 0:
    print(f"  Within-family (e.g. GAN mistaken for a different GAN):        "
          f"{within_family_errors} ({within_family_errors/total_errors:.1%})")
    print(f"  Cross-family (e.g. a GAN mistaken for a diffusion model):     "
          f"{cross_family_errors} ({cross_family_errors/total_errors:.1%})")
    print()
    if within_family_errors / total_errors > 0.7:
        print("Most errors stay within the same family -- consistent with the model having learned")
        print("real, generalizable family-level structure (GAN-ness vs diffusion-ness), with finer")
        print("within-family distinctions being the harder, noisier part. Report this finding as-is,")
        print("whichever way the actual numbers above land -- do not round it up or down to fit a story.")
    else:
        print("Errors are NOT concentrated within family -- a meaningful fraction of mistakes cross the")
        print("GAN/diffusion boundary. This would be consistent with the Modelship Attribution paper's own")
        print("caveat (cited in the Phase 2 doc) that these fingerprints don't always transfer as cleanly")
        print("as a first read of the literature suggests. Report this honestly; it's a legitimate,")
        print("citable finding about a real, open problem, not a failure of this project.")
else:
    print("No misclassifications on the test set.")

## Step 16 — Save and download

Same metadata discipline as Stage 1 (and the same bug once found and fixed there: `trained_on` must describe what ACTUALLY got used this run, not be copy-pasted from a template).

In [ ]:
import datetime

final_checkpoint = {
    "model_state_dict": model.state_dict(),
    "architecture": "efficientnet_b0",
    "class_order": GENERATOR_CLASSES,
    "input_size": INPUT_SIZE,
    "normalize_mean": NORMALIZE_MEAN,
    "normalize_std": NORMALIZE_STD,
    "trained_on": "awsaf49/artifact-dataset",
    "generator_classes_used": GENERATOR_CLASSES,
    "images_per_class": min_class_size,
    "test_accuracy": test_acc,
    "saved_at_utc": datetime.datetime.utcnow().isoformat(),
}
torch.save(final_checkpoint, CHECKPOINT_PATH)
print(f"Saved: {CHECKPOINT_PATH}")
print(f"Test accuracy: {test_acc:.4f}")

if USE_DRIVE_BACKUP:
    shutil.copy(CHECKPOINT_PATH, f"{DRIVE_BACKUP_DIR}/deepguard_attribution_FINAL.pth")
    print(f"Backed up to {DRIVE_BACKUP_DIR}/deepguard_attribution_FINAL.pth")

In [ ]:
from google.colab import files
files.download(CHECKPOINT_PATH)

## Next step (back on your laptop, not in Colab)

1. Download `deepguard_attribution.pth` from the cell above.
2. Place it at exactly: `deepguard-bouncer/models/deepguard_attribution.pth`
3. Restart your local server. The startup log should show `Attribution model loaded: 8 generator classes` instead of the "no checkpoint" message.
4. Upload an AI-generated test image through the main page — if it's flagged as "manipulated," a new "Likely source" section will appear automatically. No code changes needed on that end; it's already built and waiting for this file.